# 14 · DeepAR: Autoregressive Recurrent Networks with a Likelihood

**DeepAR** (Salinas, Flunkert, Gasthaus & Januschowski, Amazon, 2017; *IJF* 2020) is the reference deep-learning
probabilistic forecaster for large panels of related series, e.g. demand for thousands of products. It combines three ideas:
1. an **autoregressive recurrent network** (LSTM) that reads the series one step at a time, including its own last value,
2. a **parametric likelihood** at every step (NegBin for counts, Gaussian / Student-t for real values), and
3. a **global** model, trained jointly on all series, that forecasts by **sampling whole future paths**.

This notebook builds DeepAR from scratch in PyTorch on the same 300-SKU panel and backtest as nb 13 (TCN), so the
scores can be compared directly.

> **Requirements:** PyTorch (`pip install torch`, CPU build is enough). Runs in about a minute.

## Theory

### 1. The model
For series $i$ at time $t$, with known covariates $\mathbf x_{i,t}$ (calendar month, age, ...):

$$\mathbf h_{i,t}=\text{LSTM}\big(\mathbf h_{i,t-1},\ z_{i,t-1}/\nu_i,\ \mathbf x_{i,t}\big),\qquad
\theta_{i,t}=\theta(\mathbf h_{i,t}),\qquad z_{i,t}\sim\ell(\cdot\mid\theta_{i,t}).$$

The network's input includes the **previous value** $z_{t-1}$ (the "AR"). The hidden state is mapped to the parameters of
a chosen likelihood:

| Likelihood | Parameters from $\mathbf h_t$ | Use for |
|---|---|---|
| Gaussian | $\mu=\nu\,(\mathbf w^\top\mathbf h+b)$, $\sigma=\nu\,\text{softplus}(\cdot)$ | real-valued, roughly symmetric |
| Student-t | $\mu,\sigma,\nu_{\text{dof}}$ | heavy tails |
| **Negative binomial** | $\mu=\nu\,\text{softplus}(\cdot)$, $\alpha=\text{softplus}(\cdot)/\sqrt{\nu}$ | **counts**, intermittent demand (nb 08) |

NegBin parametrisation: mean $\mu$, variance $\mu+\alpha\mu^2$,

$$\log\ell(z\mid\mu,\alpha)=\log\Gamma(z+\tfrac1\alpha)-\log\Gamma(\tfrac1\alpha)-\log\Gamma(z+1)
+\tfrac1\alpha\log\tfrac{1}{1+\alpha\mu}+z\log\tfrac{\alpha\mu}{1+\alpha\mu}.$$

### 2. Scaling
Series differ by orders of magnitude. Each window is divided by $\nu_i=1+\text{mean}(z_{i,\text{context}})$ before it
enters the LSTM, and the output parameters are scaled back. For NegBin, $\mu$ is multiplied by $\nu$ and $\alpha$ is
divided by $\sqrt\nu$. The relative noise $\sqrt{\alpha}$ of a bigger series is then smaller, which is what the
Gamma–Poisson structure implies when you aggregate. $\log\nu$ is also fed in as a static covariate.

### 3. Training: maximum likelihood with teacher forcing
Sample windows of length $C+H$ (here $36+12$) from all series and maximise the log-likelihood of **every** step in the
window, feeding the **true** previous value as input ("teacher forcing"):

$$\mathcal L=\sum_{\text{windows}}\ \sum_{t}\ \log\ell\big(z_{i,t}\mid\theta(\mathbf h_{i,t})\big).$$

The original paper also samples windows with probability proportional to $\nu$, so that the few high-volume series
are not drowned out by the many slow movers. We keep that as an option (`SCALE_WEIGHTED`).

### 4. Forecasting: ancestral sampling of paths
1. Run the LSTM over the context (true values) to get the state at the forecast origin.
2. Draw $\hat z_{T+1}\sim\ell(\theta_{T+1})$, **feed the draw back in** as the next input, draw $\hat z_{T+2}$, and so on.
3. Repeat $M$ times, which gives $M$ **sample paths**.

```
context (true values) ──► h_T ──► ẑ₁ ──► ẑ₂ ──► ... ──► ẑ_H     path 1
                               └► ẑ₁ ──► ẑ₂ ──► ... ──► ẑ_H     path 2   ... path M
```

Paths are **joint across horizons**. Any functional of the future (annual total, peak month, probability of a
stock-out within 3 months) comes straight from the paths. That is impossible with per-horizon quantiles (nb 05).

### 5. DeepAR vs a direct quantile TCN (nb 13)
| | TCN + quantile head (nb 13) | **DeepAR** |
|---|---|---|
| Output | 12 × 19 quantiles in one shot | a likelihood per step, then sample paths |
| Distribution shape | free (learned from data) | fixed family (NegBin, Gaussian, ...) |
| Joint distribution over horizons | no | **yes** (paths) |
| Count data | fractional quantiles (round them) | **native**: integer samples, exact $P(Y=0)$ |
| Risk | quantile crossing (prevented by the head) | wrong family; **exposure bias** (trained on true inputs, forecasts from its own samples) |

Relatives: DeepVAR / GPVar (multivariate, Gaussian copula: nb 09), DeepState, MQ-RNN, and the foundation models
(Chronos, TimesFM, Moirai) that also output samples or distributions.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
import pf_utils as pf
pf.set_style()
try:
    import torch, torch.nn as nn, torch.nn.functional as F
except ImportError as e:
    raise ImportError("This notebook needs PyTorch: pip install torch (the CPU build is enough)") from e
rng = np.random.default_rng(14)
print("torch", torch.__version__)

torch 2.14.1


## Part A: the NegBin likelihood in PyTorch
We write the log-likelihood ourselves (it must be differentiable in $\mu$ and $\alpha$) and check it against SciPy.
SciPy uses the $(n,p)$ parametrisation: $n=1/\alpha$, $p=n/(n+\mu)$.

In [2]:
def negbin_logpmf(z, mu, alpha):
    r = 1.0 / alpha
    return (torch.lgamma(z + r) - torch.lgamma(r) - torch.lgamma(z + 1)
            + r * torch.log(r / (r + mu)) + z * torch.log(mu / (r + mu)))

z = torch.tensor([0., 1., 3., 10., 40.]); mu = torch.tensor([0.5, 2., 5., 8., 30.]); al = torch.tensor([0.1, 0.5, 1., 0.3, 0.05])
n = 1 / al.numpy()
print("max |torch - scipy| =", np.abs(negbin_logpmf(z, mu, al).numpy() - stats.nbinom.logpmf(z.numpy(), n, n / (n + mu.numpy()))).max())

max |torch - scipy| = 2.8947574826077016e-06


## Part B: data and windows
Same panel and backtest as nb 13: train on windows whose last month is before Jan 2022, validate on the Jan 2022
origin, test at **Jan 2023** and **Jan 2024** (12 months each).

In [3]:
y_df, mu_df, info = pf.make_sku_panel()
Y = y_df.to_numpy(); T, N = Y.shape
first_sale = np.argmax(Y > 0, axis=0)
C, H = 36, 12
L = C + H
TAUS = np.round(np.arange(0.05, 0.96, 0.05), 2)
VAL_ORIGIN, TEST_ORIGINS = 84, [96, 108]

def covariates(t0, n):
    """Known covariates for months t0 .. t0+n-1, shape (N, n, 3): sin/cos of the month and the SKU's age."""
    t = np.arange(t0, t0 + n)
    m = (y_df.index[0].month - 1 + t) % 12
    age = np.clip(t[None, :] - first_sale[:, None] + 1, 0, None)
    return np.stack([np.broadcast_to(np.sin(2 * np.pi * m / 12), (N, n)),
                     np.broadcast_to(np.cos(2 * np.pi * m / 12), (N, n)),
                     np.log1p(age) / np.log(T)], axis=-1)

def make_windows(origins):
    """Values z (B, C+H), covariates x (B, C+H, 3), scale nu (B,) for windows starting at o - C."""
    Z, X, NU = [], [], []
    for o in origins:
        z = Y[o - C:o + H].T
        Z.append(z); X.append(covariates(o - C, L)); NU.append(1.0 + z[:, :C].mean(axis=1))
    Z, X, NU = map(np.concatenate, (Z, X, NU))
    keep = Z[:, :C].sum(axis=1) > 0                      # skip SKUs not launched yet
    return tuple(torch.tensor(a[keep], dtype=torch.float32) for a in (Z, X, NU))

Z_tr, X_tr, NU_tr = make_windows(range(C, VAL_ORIGIN - H + 1))
Z_va, X_va, NU_va = make_windows([VAL_ORIGIN])
print(f"training windows: {len(Z_tr):,} of length {L}   validation windows: {len(Z_va)}")

training windows: 9,142 of length 48   validation windows: 298


## Part C: the model
A 2-layer LSTM with 40 hidden units. Inputs at step $t$: $z_{t-1}/\nu$, the three covariates of month $t$, and $\log\nu$.
The `likelihood` argument switches between a NegBin and a Gaussian head, so we can compare the two.

In [4]:
class DeepAR(nn.Module):
    def __init__(self, likelihood="negbin", hidden=40, layers=2, dropout=0.1):
        super().__init__()
        self.likelihood = likelihood
        self.rnn = nn.LSTM(1 + 3 + 1, hidden, layers, batch_first=True, dropout=dropout)
        self.proj = nn.Linear(hidden, 2)

    def inputs(self, z_prev, x, nu):
        """Stack [z_{t-1}/nu, covariates_t, log nu] into the LSTM input (B, n, 5)."""
        log_nu = torch.log(nu)[:, None, None].expand(-1, x.shape[1], 1)
        return torch.cat([(z_prev / nu[:, None])[..., None], x, log_nu], dim=-1)

    def distribution_params(self, h, nu):
        a, b = self.proj(h).unbind(-1)
        nu = nu[:, None]
        if self.likelihood == "negbin":
            return nu * F.softplus(a) + 1e-6, F.softplus(b) / torch.sqrt(nu) + 1e-6      # mu, alpha
        return nu * a, nu * F.softplus(b) + 1e-6                                          # mean, sd

    def log_lik(self, z, p1, p2):
        if self.likelihood == "negbin":
            return negbin_logpmf(z, p1, p2)
        return torch.distributions.Normal(p1, p2).log_prob(z)

    def loss(self, z, x, nu):
        """Mean negative log-likelihood over steps 2..C+H of each window (teacher forcing)."""
        z_prev = torch.cat([torch.zeros_like(z[:, :1]), z[:, :-1]], dim=1)
        h, _ = self.rnn(self.inputs(z_prev, x, nu))
        p1, p2 = self.distribution_params(h[:, 1:], nu)
        return -self.log_lik(z[:, 1:], p1, p2).mean()

    @torch.no_grad()
    def sample_paths(self, z_ctx, x_all, nu, M=200):
        """Ancestral sampling: returns (B, M, H) sample paths after conditioning on z_ctx (B, C)."""
        B = z_ctx.shape[0]
        z_prev = torch.cat([torch.zeros_like(z_ctx[:, :1]), z_ctx[:, :-1]], dim=1)
        _, state = self.rnn(self.inputs(z_prev, x_all[:, :C], nu))                     # read the context
        state = tuple(s.repeat_interleave(M, dim=1) for s in state)                     # M copies per series
        nu_m, x_m, last = nu.repeat_interleave(M), x_all.repeat_interleave(M, 0), z_ctx[:, -1:].repeat_interleave(M, 0)
        paths = []
        for h in range(H):
            out, state = self.rnn(self.inputs(last, x_m[:, C + h:C + h + 1], nu_m), state)
            p1, p2 = self.distribution_params(out, nu_m)
            if self.likelihood == "negbin":                                             # Gamma-Poisson draw
                r = 1 / p2
                draw = torch.poisson(torch.distributions.Gamma(r, r / p1).sample())
            else:
                draw = torch.distributions.Normal(p1, p2).sample()
            paths.append(draw)
            last = draw                                                                 # feed the sample back in
        return torch.cat(paths, dim=1).view(B, M, H).numpy()

### Training
Uniform window sampling by default. With `SCALE_WEIGHTED = True` windows are drawn with probability $\propto\nu$, as in
the paper. On this panel uniform sampling was slightly better: over three seeds, wQL ≈ 0.41 with uniform sampling and
≈ 0.43 with scale-weighted sampling.

In [5]:
EPOCHS, BATCH, SCALE_WEIGHTED = 25, 256, False

def train(likelihood, seed=14):
    torch.manual_seed(seed)
    model = DeepAR(likelihood)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    w = NU_tr / NU_tr.sum()
    hist = []
    for ep in range(EPOCHS):
        model.train()
        order = torch.multinomial(w, len(w), replacement=True) if SCALE_WEIGHTED else torch.randperm(len(w))
        tot = 0.0
        for b in range(0, len(order), BATCH):
            i = order[b:b + BATCH]
            loss = model.loss(Z_tr[i], X_tr[i], NU_tr[i])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            opt.step()
            tot += loss.item() * len(i)
        model.eval()
        with torch.no_grad():
            hist.append({"epoch": ep + 1, "training": tot / len(order), "validation": model.loss(Z_va, X_va, NU_va).item()})
    return model, pd.DataFrame(hist).set_index("epoch")

t0 = time.time()
models, hists = {}, {}
for lik in ["negbin", "gaussian"]:
    models[lik], hists[lik] = train(lik)
print(f"trained 2 models in {time.time() - t0:.0f} s")

fig = pf.subplots(1, 2, titles=["NegBin likelihood", "Gaussian likelihood"], height=360)
for c, lik in enumerate(["negbin", "gaussian"], start=1):
    pf.add_lines_by(fig, hists[lik], {"training": pf.SERIES[0], "validation": pf.SERIES[1]}, row=1, col=c,
                    showlegend=c == 1, fmt=".3f", markers=False)
fig.update_layout(title="Negative log-likelihood per step (the two panels are not comparable: pmf vs density)",
                  hovermode="x unified")
fig.update_xaxes(title_text="epoch")
fig.update_yaxes(title_text="mean NLL", col=1)
fig.show()

trained 2 models in 52 s


## Part D: sample paths, quantiles and scores
For each test origin we condition on the latest 36 months and draw $M=200$ paths per SKU. Quantiles are read off the
paths. The Gaussian model can produce negative demand, so we clip its quantiles at 0. Baselines and scores are
exactly those of nb 13.

In [6]:
def deepar_paths(model, o, M=200):
    z_ctx = torch.tensor(Y[o - C:o].T, dtype=torch.float32)
    x_all = torch.tensor(covariates(o - C, L), dtype=torch.float32)
    torch.manual_seed(o)
    return model.eval().sample_paths(z_ctx, x_all, 1.0 + z_ctx.mean(dim=1), M)          # (N, M, H)

PATHS = {lik: np.concatenate([deepar_paths(models[lik], o) for o in TEST_ORIGINS]) for lik in models}
to_q = lambda S: np.moveaxis(np.quantile(S, TAUS, axis=1), 0, -1)                       # (B, H, K)

def oracle_quantiles(o):
    m = mu_df.to_numpy()[o:o + H].T; n = 1 / info.alpha.to_numpy()[:, None]
    return stats.nbinom.ppf(TAUS[None, None, :], n[..., None], (n / (n + m))[..., None])

Q = {"Seasonal naive": np.concatenate([pf.seasonal_naive_quantiles(Y, o, H, TAUS) for o in TEST_ORIGINS]),
     "Local NegBin GLM": np.concatenate([pf.local_negbin_quantiles(Y, o, H, TAUS) for o in TEST_ORIGINS]),
     "DeepAR, Gaussian": np.maximum(to_q(PATHS["gaussian"]), 0),
     "DeepAR, NegBin": to_q(PATHS["negbin"]),
     "Oracle (true μ)": np.concatenate([oracle_quantiles(o) for o in TEST_ORIGINS])}
COLOR = {"Seasonal naive": pf.SERIES[3], "Local NegBin GLM": pf.SERIES[1], "DeepAR, Gaussian": pf.SERIES[4],
         "DeepAR, NegBin": pf.SERIES[0], "Oracle (true μ)": pf.INK["muted"]}

Y_test = np.concatenate([Y[o:o + H].T for o in TEST_ORIGINS])
scale = np.concatenate([[Y[first_sale[i]:o, i].mean() + 1 for i in range(N)] for o in TEST_ORIGINS])
short = np.concatenate([o - first_sale for o in TEST_ORIGINS]) < 36
nu_test = np.concatenate([1 + Y[o - C:o].mean(axis=0) for o in TEST_ORIGINS])

def score_table(Q, mask=slice(None)):
    return pd.DataFrame({n: pf.panel_quantile_scores(Y_test[mask], q[mask], TAUS, scale[mask]) for n, q in Q.items()}).T

display(score_table(Q))
pd.DataFrame({f"sQL, history < 36 months (n={short.sum()})": score_table(Q, short)["sQL"],
              f"sQL, history ≥ 36 months (n={(~short).sum()})": score_table(Q, ~short)["sQL"]})

,wQL,sQL,cov80,cov90
Seasonal naive,0.555,0.550,0.705,0.819
Local NegBin GLM,0.427,0.435,0.811,0.897
"DeepAR, Gaussian",0.413,0.418,0.858,0.925
"DeepAR, NegBin",0.409,0.404,0.831,0.905
Oracle (true μ),0.333,0.340,0.853,0.932


,"sQL, history < 36 months (n=56)","sQL, history ≥ 36 months (n=544)"
Seasonal naive,0.720,0.532
Local NegBin GLM,0.771,0.400
"DeepAR, Gaussian",0.480,0.411
"DeepAR, NegBin",0.476,0.397
Oracle (true μ),0.397,0.334


**Reading the scores**
* DeepAR (NegBin) beats both local baselines and lands in the same range as the TCN of nb 13 (wQL ≈ 0.40 there). As
  with the TCN, the gain comes from the **short-history SKUs**: the global model has learned launch ramps and seasonal
  families from the other series. On long histories it is level with the local NegBin GLM.
* Unlike the raw TCN, DeepAR's intervals are **calibrated out of the box** (coverage at or slightly above nominal, as
  for any discrete forecast). The NegBin family is the right shape for this data, and its samples are integers.
* The Gaussian DeepAR is only slightly worse on these averaged quantile scores, which are dominated by the many
  mid-size SKUs where a Gaussian is a fair approximation. Its intervals over-cover. Part E shows where the wrong
  family really hurts.

In [7]:
o_pos, idx = 1, y_df.index[TEST_ORIGINS[1]:TEST_ORIGINS[1] + H]                        # the Jan 2024 origin
show = {"holiday, fast mover": info.index[(info.family == "holiday") & (info.level > 60)][0],
        "school, launched 2020": info.index[(info.family == "school") & (info.launch.dt.year == 2020)][0],
        "spring, slow mover": info.index[(info.family == "spring") & (info.level < 1.5)][0]}
fig = pf.subplots(1, 3, titles=list(show), height=440)
for c, (label, sku) in enumerate(show.items(), start=1):
    i = o_pos * N + y_df.columns.get_loc(sku)
    S = PATHS["negbin"][i]
    pf.add_paths(fig, idx, S, color=pf.SERIES[0], name="sample paths", n=30, row=1, col=c, showlegend=c == 1, opacity=0.25)
    qd = {tau: np.quantile(S, tau, axis=0) for tau in [0.05, 0.5, 0.95]}
    pf.add_fan(fig, idx, qd, history=y_df[sku].iloc[:TEST_ORIGINS[1]], actual=Y_test[i], n_history=24,
               label="DeepAR", row=1, col=c, showlegend=c == 1, fmt=",.0f")
fig.update_layout(title="DeepAR (NegBin): 30 of the 200 sample paths and the 90% interval for 2024",
                  legend=dict(orientation="h", y=-0.15), margin=dict(b=90))
fig.update_yaxes(title_text="units", col=1)
fig.show()

Each path is one coherent scenario for the whole year: an integer number of units every month, including zero months
for the slow mover.

## Part E: where the likelihood matters, zero months
For stock decisions on slow movers the key question is often *"will we sell anything at all?"*. A NegBin model gives
$P(Y=0)$ directly. For the Gaussian model, the nearest equivalent is $P(Y<0.5)$, which includes all the probability
it puts on **negative demand**.

In [8]:
p0 = {"DeepAR, NegBin": (PATHS["negbin"] == 0).mean(axis=1).ravel(),
      "DeepAR, Gaussian": (PATHS["gaussian"] < 0.5).mean(axis=1).ravel()}
is_zero = (Y_test == 0).ravel()
print(f"observed share of zero months: {is_zero.mean():.3f}")
for name, p in p0.items():
    print(f"{name:18s} mean predicted P(zero) = {p.mean():.3f}   Brier = {pf.brier_score(is_zero, p):.4f}")
print(f"Gaussian paths with negative demand: {(PATHS['gaussian'] < 0).mean():.1%} of all simulated months")

fig = go.Figure()
for k, (name, p) in enumerate(p0.items()):
    pf.add_reliability(fig, is_zero, p, n_bins=8, name=name, color=COLOR[name], diagonal=k == 0)
fig.update_layout(title="Reliability of P(no sales this month)", height=440, width=560)
fig.show()

observed share of zero months: 0.097
DeepAR, NegBin     mean predicted P(zero) = 0.089   Brier = 0.0635
DeepAR, Gaussian   mean predicted P(zero) = 0.126   Brier = 0.0689
Gaussian paths with negative demand: 9.5% of all simulated months


The NegBin curve stays close to the diagonal across the whole range. The Gaussian curve is S-shaped, and wrong in
both directions:
* For **slow movers**, where zero months are common, it *under*-predicts them (hover the upper bins: forecasts around
  0.4–0.7 against observed frequencies around 0.7–0.9). A symmetric distribution cannot be *probably zero but
  occasionally large*, which is what intermittent demand looks like (nb 08).
* For **mid-size SKUs** it *over*-predicts "zero or less" (the low bins), and that probability is mostly the
  impossible negative demand.
For a stocking decision on slow movers, this is the difference that matters, even though the averaged wQL barely moved.

## Part F: path functionals, the annual total
Suppose you need a 90% interval for each SKU's **total 2023 (or 2024) demand**, e.g. for a yearly purchase contract.
* **From the paths:** sum each path over the 12 months, then take the 5% and 95% quantiles of those totals.
* **Wrong shortcut:** add up the monthly 5% and 95% quantiles. Quantiles are not additive. The sum of monthly 5%
  quantiles is far below the 5% quantile of the sum, because it assumes every month is simultaneously at its 5% level.

In [9]:
tot_actual = Y_test.sum(axis=1)
S_tot = PATHS["negbin"].sum(axis=2)                                         # (B, M) annual totals per path
lo_p, hi_p = np.quantile(S_tot, [0.05, 0.95], axis=1)
q05, q95 = Q["DeepAR, NegBin"][..., list(TAUS).index(0.05)], Q["DeepAR, NegBin"][..., list(TAUS).index(0.95)]
lo_s, hi_s = q05.sum(axis=1), q95.sum(axis=1)
pd.DataFrame({"90% coverage": [pf.coverage(tot_actual, lo_p, hi_p), pf.coverage(tot_actual, lo_s, hi_s)],
              "median width / actual total": [np.median((hi_p - lo_p) / np.maximum(tot_actual, 1)),
                                              np.median((hi_s - lo_s) / np.maximum(tot_actual, 1))],
              "CRPS of the total (scaled)": [np.mean(pf.crps_sample(tot_actual, S_tot.T) / (12 * scale)), np.nan]},
             index=["quantiles of summed paths", "sum of monthly quantiles ✗"])

,90% coverage,median width / actual total,CRPS of the total (scaled)
quantiles of summed paths,0.808,0.842,0.167
sum of monthly quantiles ✗,0.997,2.367,NaN


In [10]:
fig = pf.subplots(1, 1, titles=["PIT of the annual total, from the DeepAR (NegBin) paths"], height=340)
pf.add_pit(fig, pf.pit_from_samples(tot_actual, S_tot.T), bins=10)
fig.update_layout(width=560)
fig.show()

* The **sum of monthly quantiles** covers almost 100% and is almost three times as wide as the path-based interval: useless for a
  contract. Quantiles do not add up.
* The **path-based** interval is far sharper but **under-covers** (about 81% for a nominal 90%), although each month on
  its own was calibrated (Part D). The PIT of the annual total shows why: too many actual totals fall in the two
  outer bins, especially the lowest. The distributions of the yearly totals are too narrow, and for some SKUs too high. A single trained network carries no
  uncertainty about its own level estimate (epistemic uncertainty, nb 11), and errors in that level are shared by
  all 12 months of a path, so they add up in the total.
* That is the general lesson of nb 09: **marginal calibration does not imply joint calibration**. Always check the
  functionals you will actually use. Practical fixes: an ensemble of networks (nb 05), or a conformal correction
  of the annual totals on the validation origin (nb 04).
* Paths answer *any* question about the future year with one set of samples: annual totals, P(any stock-out),
  the month of the peak, cumulative demand until a reorder arrives.

## Takeaways
* DeepAR = **autoregressive LSTM + per-step likelihood + global training + ancestral sampling**.
* Pick the likelihood to match the data. NegBin for counts gives calibrated intervals and exact $P(Y=0)$, while a
  Gaussian puts mass on negative demand and misses the zeros.
* **Scaling** by $\nu$ (and $\alpha/\sqrt\nu$) lets one network serve SKUs from 0.5 to 500 units per month.
* **Sample paths** are joint across horizons. Use them for totals, peaks and stock-out probabilities, never add up
  marginal quantiles, and check the calibration of the functional itself: calibrated months do not guarantee a
  calibrated annual total.
* Like the TCN, DeepAR beats local models most where there is little history to learn from.
* Libraries: GluonTS `DeepAREstimator` (reference implementation), `pytorch-forecasting` `DeepAR`, Nixtla
  `neuralforecast.models.DeepAR`, Darts `RNNModel(likelihood=...)`, Amazon SageMaker's built-in DeepAR.

### Exercises
1. Set `SCALE_WEIGHTED = True` and retrain with a few seeds. Does it help the big SKUs (wQL) at the expense of the small
   ones (sQL)?
2. Replace the NegBin head with a Student-t head (`torch.distributions.StudentT`) and compare with the Gaussian one.
3. **Exposure bias:** compare the wQL at $h=1$ with $h=12$ for DeepAR and for the TCN of nb 13 (which never feeds its
   own forecasts back in). Which degrades faster?
4. **Cold start:** for SKUs launched in the last 6 months before the origin, the context is mostly zeros. Inspect their
   forecasts. Is the ramp-up learned?
5. Train 5 DeepAR models with different seeds and pool their paths (a deep ensemble). Does the coverage of the
   annual total move towards 90%?
6. Estimate P(total 2024 demand > 1.2 × total 2023 demand) for every SKU from the paths, and check its reliability
   diagram against the outcomes (nb 06).